# Session 7 — Text classification at scale, with validation

The full loop, on our synthetic survey: **codebook → batch classify → validate against a human → agreement metrics → error analysis.** This notebook is deliberately the skeleton of Assignment 4, Option 1.

In [ ]:
# Thirty-second data check -- the course datasets were replaced on 10 July.
import pandas as pd
sample = pd.read_csv("../data/sample_200.csv")
if sample.resp_id.iloc[0] == "R0251" and len(sample) == 200:
    print("Data check OK -- you are on the current dataset.")
else:
    print("!!! STALE DATA -- your repo predates the 11 July data update.")
    print("    Run in the terminal:  bash sync_data.sh   (repo root, one command)")
    print("    Then: Kernel -> Restart, and rerun this cell before continuing.")

In [ ]:
import sys, pathlib, time
sys.path.append(str(pathlib.Path("..").resolve()))   # notebooks/ -> repo root
import course_utils

try:
    client = course_utils.get_client(verbose=True)
except RuntimeError:
    # First time here without check_setup? We can store the key now too:
    import getpass, os
    key = getpass.getpass("Paste your API key (input stays hidden): ")
    course_utils._persist("GEMINI_API_KEY", key)
    os.environ["GEMINI_API_KEY"] = key
    client = course_utils.get_client(verbose=True)

MODEL = course_utils.MODEL
import os

# Polite-API pattern: the free tier meters ~10 requests/minute, so loops in this
# notebook pace themselves, and a rate-limit error ("429") waits and retries once
# instead of crashing. Steal this wrapper for your own projects.
PACE = 6.5 if os.environ.get("GEMINI_MODE") == "dev" else 1.0  # seconds between calls

def ask(contents, **config):
    for attempt in (1, 2):
        try:
            return client.models.generate_content(model=MODEL, contents=contents,
                                                  config=config or None)
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                if attempt == 1:
                    print("   (rate limit reached -- waiting 30s and retrying...)")
                    time.sleep(30)
                else:
                    raise RuntimeError("Still rate-limited. Wait a minute, then rerun this cell. "
                                       "This is the meter, not a broken key.") from None
            else:
                raise

print("Client ready.")

## 1. Load the fixed sample and the codebook

One rule to write down before classifying: the codebook has five categories and no escape hatch — our documented rule today is that texts fitting no category get `UNCLEAR`.

In [ ]:
import json
codebook = open("../data/codebook_concerns.md", encoding="utf-8").read()
print(len(sample), "texts"); sample.head(3)

## 2. Classify in batches of 25

Watch the pattern: numbered batch in → JSON list out → collected into one table. (In-session we do the first 100 rows; scaling to all 200 is your take-home.)

In [ ]:
def classify_batch(texts, start_n, instruction=None):
    instruction = instruction or ("Using ONLY the codebook below, assign exactly one category "
                                  "(JOB/SURV/SKILL/FAIR/REL) to each numbered text. "
                                  "If none fits, use UNCLEAR.")
    numbered = "\n".join(f"{start_n+i}. {t}" for i, t in enumerate(texts))
    r = ask(f"""{instruction}
Return a JSON list of objects with keys: n, category.
CODEBOOK:\n{codebook}\n\nTEXTS:\n{numbered}""",
            temperature=0, response_mime_type="application/json")
    return json.loads(r.text)

results = []
N = 100  # in-session scope; set to len(sample) at home
for start in range(0, N, 25):
    chunk = sample.concern_text[start:start+25].tolist()
    results += classify_batch(chunk, start + 1)
    print(f"rows {start+1}-{start+len(chunk)} done")
    time.sleep(PACE)
labels = pd.DataFrame(results).sort_values("n").reset_index(drop=True)
labels["category"].value_counts()

## 3. Validate against a human: you

Before looking at the model's answers for rows 1–20, code them yourself with only the codebook (pen and paper is fine). The next cell prints a clean coding sheet — texts only, no model labels. *No peeking — the whole point dies with a peek.*

In [ ]:
# Your coding sheet -- rows 1-20, texts only:
for i, t in enumerate(sample.concern_text[:20], start=1):
    print(f"{i:2d}. {t}")

In [ ]:
my_labels = []  # e.g. ["JOB", "REL", ...]  <- your 20 codes for rows 1-20, in order
if len(my_labels) == 20:
    from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score
    model_20 = labels.category[:20].tolist()
    print("agreement:", accuracy_score(my_labels, model_20))
    print("Cohen's kappa:", round(cohen_kappa_score(my_labels, model_20), 2))
    cats = sorted(set(my_labels) | set(model_20))
    print(pd.DataFrame(confusion_matrix(my_labels, model_20, labels=cats), index=cats, columns=cats))
else:
    print(f"my_labels has {len(my_labels)} entries -- it needs exactly 20 (rows 1-20, codebook only).")

## 4. Error analysis — the part that makes it research

For every disagreement: who was wrong, you or the model — or is the *codebook* ambiguous? (Session discussion; in A4 this becomes your written error table.)

In [ ]:
if len(my_labels) == 20:
    side = pd.DataFrame({"text": sample.concern_text[:20], "you": my_labels,
                         "model": labels.category[:20]})
    disagreements = side[side.you != side.model]
    print(f"{len(disagreements)} disagreement(s):")
    with pd.option_context("display.max_colwidth", 100):
        display(disagreements)
else:
    print("Fill in my_labels above first.")

## 5. Robustness probe

Rerun rows 1–50 with the instruction sentence paraphrased — does the model agree with itself? A classifier that changes its mind when the prompt is reworded is measuring the prompt, not the construct.

In [ ]:
reworded = ("Read each numbered text and label it with the single best-fitting code "
            "(JOB/SURV/SKILL/FAIR/REL) as defined in the codebook below; "
            "use UNCLEAR only when no code applies.")
results_v2 = []
for start in range(0, 50, 25):
    chunk = sample.concern_text[start:start+25].tolist()
    results_v2 += classify_batch(chunk, start + 1, instruction=reworded)
    time.sleep(PACE)
v2 = pd.DataFrame(results_v2).sort_values("n").reset_index(drop=True)
same = (labels.category[:50].values == v2.category[:50].values).mean()
print(f"Self-agreement under paraphrase, rows 1-50: {same:.0%}")

**Take-home:** set `N = len(sample)` in section 2 (4 more requests) and rerun sections 3–5 on the full sample — that's Session 7 as a complete dress rehearsal for A4 Option 1.